# Notebook 11 — Functional Testing of the Artefact

**Project:** A Multi-Modal Deep Learning Approach for Eggshell Micro-Crack Morphology Analysis and Root-Cause Diagnosis in Poultry Breeder Farms
**Author:** M.P.N. Sharindi (Coventry Index 16110429)
**Stage:** 11 of 12 — Section 5.4

---

### What this notebook is, and what it is not

The self-check guidelines are explicit: functional testing **confirms the artefact works as built**
and is *not* the research result. It is kept short here, with the full test table going to an
appendix, and nothing in it is presented as evidence about crack detection.

Everything measured so far was measured in notebooks, on a cached array, through code paths that
no end user touches. The dashboard is a different program. It reads a JPEG from disk, segments the
egg again, fills the background again, resizes again, and runs the model on CPU. **None of that has
been shown to reproduce the pipeline the reported numbers came from**, and a mismatch there would
not raise an error — it would quietly return confident wrong answers.

That is the main risk this notebook retires.

| test | question |
|---|---|
| T1 | Does the app's preprocessing reproduce the training cache pixel for pixel? |
| T2 | Does the app reproduce notebook 07's test-set predictions? |
| T3 | Is inference deterministic across repeated calls? |
| T4 | Does the app degrade gracefully on malformed input? |
| T5 | Is the rule base complete and does every cell return usable text? |
| T6 | Do the threshold and the uncertainty flag behave as specified? |
| T7 | What is CPU latency, which is what a farm laptop will actually see? |

### A known difference, measured rather than assumed

`neutral_fill` adds noise to the replaced background, seeded per image. Notebook 02 seeded with the
row index; the app has no row index and seeds with 0. The images therefore differ by a small amount
of background noise. T1 and T2 quantify whether that matters. It should not, but "should not" is
not a test result.

---
## 0. Setup

In [1]:
import os
os.environ["EGG_ARTEFACT_ROOT"] = r"D:\Eggshell-Diagnostic-Research\Artefact"

N_PIXEL_CHECK = 40      # eggs for the pixel-level comparison
TRAIN_SIDE = 384
BACKBONE, FORMULATION = "efficientnet_b0", "multitask_seg"
SEEDS = [42, 43, 44]
DEFAULT_THRESHOLD, DEFAULT_LOW_CONF = 0.30, 0.60

In [2]:
import sys, json, time, warnings, traceback
from pathlib import Path
from datetime import datetime
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / "src" / "common").is_dir()), None)
if _root is None:
    raise RuntimeError("Run from Artefact/notebooks/ or set EGG_ARTEFACT_ROOT.")
sys.path.insert(0, str(_root / "src")); sys.path.insert(0, str(_root / "app"))

import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
import torch, torch.nn.functional as F

from common import project_config as CFG
from common.models import build_model
from common.preprocess import preprocess_bgr, to_model_input

warnings.filterwarnings("ignore")
STAGE = "11_functional_testing"
OUT = CFG.results_dir(STAGE); FIG_DIR, TAB_DIR = OUT / "figures", OUT / "tables"
PROC = CFG.DATA_DIR / "processed"
PRED07 = CFG.RESULTS_DIR / "07_final_and_test" / "predictions"

splits = pd.read_csv(PROC / "splits.csv")
cache = np.load(PROC / "cache_neutral_fill_448.npy", mmap_mode="r")
device = torch.device("cpu")      # the farm laptop case, and what Streamlit uses by default

def load_models(dev):
    d = CFG.MODELS_DIR / "final"; out = []
    for s in SEEDS:
        p = d / f"final_{BACKBONE}_{FORMULATION}_seed{s}.pt"
        if not p.exists():
            continue
        sd = torch.load(p, map_location="cpu", weights_only=False)
        m = build_model(BACKBONE, FORMULATION, 4, pretrained=False, age_mode="none",
                        img_size=TRAIN_SIDE)
        m.load_state_dict(sd["state_dict"]); m.eval(); m.to(dev); out.append(m)
    return out

MODELS = load_models(device)
print(f"{len(MODELS)} model checkpoints loaded on {device}")

@torch.no_grad()
def app_infer(rgb448):
    """Exactly the inference path the dashboard uses."""
    x = torch.from_numpy(to_model_input(rgb448, TRAIN_SIDE)).to(device)
    p4s, segs = [], []
    for m in MODELS:
        feats = m.backbone(x)
        pooled = m.drop(F.adaptive_avg_pool2d(feats[-1], 1).flatten(1))
        det = torch.softmax(m.head_det(pooled), 1)
        pat = torch.softmax(m.head_pat(pooled), 1)
        p4s.append(torch.cat([det[:, :1], det[:, 1:2] * pat], 1)[0].cpu().numpy())
        segs.append(torch.sigmoid(m.seg_head(feats))[0, 0].cpu().numpy())
    return np.mean(p4s, axis=0), np.mean(segs, axis=0)

RESULTS = []
def record(tid, name, passed, detail=""):
    RESULTS.append({"id": tid, "test": name, "result": "PASS" if passed else "FAIL",
                    "detail": str(detail)[:200]})
    print(f"  [{'PASS' if passed else 'FAIL'}] {tid} {name} — {detail}")

3 model checkpoints loaded on cpu


---
## T1 · Does the app's preprocessing reproduce the training cache?

In [3]:
def resolve_path(row):
    for col in ("path", "file_name", "filename", "file"):
        if col in splits.columns and isinstance(splits.loc[row, col], str):
            p = Path(splits.loc[row, col])
            if p.is_absolute() and p.is_file():
                return p
            q = CFG.IMAGES_DIR / p.name
            if q.is_file():
                return q
    q = CFG.IMAGES_DIR / f"{splits.loc[row, 'egg_id']}.jpg"
    return q if q.is_file() else None

rng = np.random.default_rng(0)
test_rows = splits.index[splits["split"].eq("test")].to_numpy()
sample = rng.choice(test_rows, size=min(N_PIXEL_CHECK, len(test_rows)), replace=False)

diffs, miss = [], 0
for r in sample:
    p = resolve_path(int(r))
    if p is None:
        miss += 1; continue
    raw = cv2.imread(str(p), cv2.IMREAD_COLOR)
    app_img = preprocess_bgr(raw)
    if app_img is None:
        miss += 1; continue
    ref = np.asarray(cache[int(r)])
    d = np.abs(app_img.astype(np.int16) - ref.astype(np.int16))
    diffs.append({"egg_id": splits.loc[int(r), "egg_id"],
                  "mean_abs_diff": float(d.mean()), "p99_abs_diff": float(np.percentile(d, 99)),
                  "max_abs_diff": float(d.max()),
                  "corr": float(np.corrcoef(app_img.ravel(), ref.ravel())[0, 1])})
pix = pd.DataFrame(diffs)
if len(pix):
    pix.to_csv(TAB_DIR / "t01_pixel_agreement.csv", index=False)
    print(pix.describe().loc[["mean", "max"]].round(3).to_string())
    ok = bool(pix["corr"].min() > 0.99 and pix["mean_abs_diff"].mean() < 6.0)
    record("T1", "app preprocessing reproduces the cache",
           ok, f"mean|diff| {pix['mean_abs_diff'].mean():.2f}, "
               f"min corr {pix['corr'].min():.4f}, {miss} unreadable")
    print("\nA small non-zero difference is expected and explained: neutral_fill seeds its"
          "\nbackground noise per image, and the app has no row index to seed with. What matters"
          "\nis that the egg itself is identical and the correlation is near 1.")
else:
    record("T1", "app preprocessing reproduces the cache", False,
           "no original images found; set EGG_IMAGES_DIR")

      mean_abs_diff  p99_abs_diff  max_abs_diff  corr
mean          0.677          6.35        15.675   1.0
max           1.864         18.00        46.000   1.0
  [PASS] T1 app preprocessing reproduces the cache — mean|diff| 0.68, min corr 0.9979, 0 unreadable

A small non-zero difference is expected and explained: neutral_fill seeds its
background noise per image, and the app has no row index to seed with. What matters
is that the egg itself is identical and the correlation is near 1.


---
## T2 · Does the app reproduce notebook 07's predictions?

In [4]:
Z = np.load(PRED07 / "test_selected.npz")
ref_idx, ref_y4, ref_prob = Z["idx"], Z["y4"], Z["prob"]
ref_pos = {int(v): i for i, v in enumerate(ref_idx)}

rows = []
for r in sample:
    p = resolve_path(int(r))
    if p is None or int(r) not in ref_pos:
        continue
    img = preprocess_bgr(cv2.imread(str(p), cv2.IMREAD_COLOR))
    if img is None:
        continue
    p4, _ = app_infer(img)
    rp = ref_prob[ref_pos[int(r)]]
    rows.append({"egg_id": splits.loc[int(r), "egg_id"],
                 "app_label": CFG.LABEL4[int(p4.argmax())],
                 "nb07_label": CFG.LABEL4[int(rp.argmax())],
                 "agree": bool(p4.argmax() == rp.argmax()),
                 "prob_mae": float(np.abs(p4 - rp).mean()),
                 "app_pcrack": float(1 - p4[0]), "nb07_pcrack": float(1 - rp[0])})
agree = pd.DataFrame(rows)
if len(agree):
    agree.to_csv(TAB_DIR / "t02_prediction_agreement.csv", index=False)
    rate = float(agree["agree"].mean())
    mae = float(agree["prob_mae"].mean())
    print(agree.head(10).to_string(index=False))
    print(f"\nlabel agreement {rate:.1%} over {len(agree)} eggs | mean probability MAE {mae:.4f}")
    record("T2", "app reproduces notebook 07 predictions", bool(rate >= 0.97 and mae < 0.02),
           f"agreement {rate:.1%}, prob MAE {mae:.4f}")
    if rate < 1.0:
        print("\ndisagreements:")
        print(agree[~agree["agree"]].to_string(index=False))
        print("Inspect these: a disagreement on a borderline egg is tolerable, a systematic")
        print("one means the two pipelines have genuinely diverged.")
else:
    record("T2", "app reproduces notebook 07 predictions", False, "no comparable eggs")

egg_id app_label nb07_label  agree  prob_mae  app_pcrack  nb07_pcrack
  E507    Intact     Intact   True  0.000753    0.125495     0.126570
  E332    Intact     Intact   True  0.006358    0.115374     0.128090
  E511      Star       Star   True  0.001441    0.942098     0.942624
  E635    Intact     Intact   True  0.003713    0.045231     0.052658
  E452  Hairline   Hairline   True  0.010056    0.551810     0.571421
  E059    Intact     Intact   True  0.000141    0.003677     0.003812
  E005    Intact     Intact   True  0.000158    0.002694     0.003007
  E523   Network    Network   True  0.012440    0.973074     0.972702
  E439      Star       Star   True  0.001115    0.979743     0.977877
  E126    Intact     Intact   True  0.001622    0.019216     0.016143

label agreement 100.0% over 40 eggs | mean probability MAE 0.0034
  [PASS] T2 app reproduces notebook 07 predictions — agreement 100.0%, prob MAE 0.0034


---
## T3–T4 · Determinism and graceful degradation

In [5]:
# T3 determinism
p = resolve_path(int(sample[0])) if len(sample) else None
if p is not None:
    img = preprocess_bgr(cv2.imread(str(p), cv2.IMREAD_COLOR), seed=0)
    a, _ = app_infer(img); b, _ = app_infer(img)
    same_img = preprocess_bgr(cv2.imread(str(p), cv2.IMREAD_COLOR), seed=0)
    record("T3a", "inference is deterministic", bool(np.allclose(a, b, atol=1e-6)),
           f"max |diff| {np.abs(a - b).max():.2e}")
    record("T3b", "preprocessing is deterministic for a fixed seed",
           bool(np.array_equal(img, same_img)), "byte-identical")
else:
    record("T3a", "inference is deterministic", False, "no image available")

# T4 malformed input
cases = {
    "1x1 pixel": np.full((1, 1, 3), 128, np.uint8),
    "tiny 12x9": np.full((9, 12, 3), 90, np.uint8),
    "uniform black": np.zeros((400, 400, 3), np.uint8),
    "uniform white": np.full((400, 400, 3), 255, np.uint8),
    "pure noise": (np.random.default_rng(1).integers(0, 255, (400, 400, 3))).astype(np.uint8),
    "extreme aspect 1200x80": np.full((80, 1200, 3), 120, np.uint8),
    "very large 5000x5000": np.full((5000, 5000, 3), 100, np.uint8),
}
rows = []
for name, arr in cases.items():
    try:
        out = preprocess_bgr(arr)
        if out is None:
            rows.append({"case": name, "outcome": "rejected cleanly (None)", "crashed": False})
        else:
            p4, _ = app_infer(out)
            rows.append({"case": name,
                         "outcome": f"ran, p(crack)={1 - p4[0]:.2f}", "crashed": False})
    except Exception as e:
        rows.append({"case": name, "outcome": f"{type(e).__name__}: {e}", "crashed": True})
edge = pd.DataFrame(rows)
edge.to_csv(TAB_DIR / "t03_edge_cases.csv", index=False)
print(edge.to_string(index=False))
record("T4", "malformed input does not crash the app",
       bool(not edge["crashed"].any()), f"{int(edge['crashed'].sum())} crashes of {len(edge)}")
print("\nNote: 'ran' on a non-egg image is not a failure of this test but IS a product gap —")
print("the app has no reject option for images that are not candled eggs. Record it in 4.9.3")
print("and in the limitations.")

# corrupted bytes, the realistic upload failure
try:
    bad = cv2.imdecode(np.frombuffer(b"not an image at all", np.uint8), cv2.IMREAD_COLOR)
    record("T4b", "corrupted bytes return None rather than raising", bad is None, "imdecode None")
except Exception as e:
    record("T4b", "corrupted bytes return None rather than raising", False, repr(e))

  [PASS] T3a inference is deterministic — max |diff| 0.00e+00
  [PASS] T3b preprocessing is deterministic for a fixed seed — byte-identical
                  case            outcome  crashed
             1x1 pixel ran, p(crack)=0.33    False
             tiny 12x9 ran, p(crack)=0.33    False
         uniform black ran, p(crack)=0.33    False
         uniform white ran, p(crack)=0.33    False
            pure noise ran, p(crack)=0.43    False
extreme aspect 1200x80 ran, p(crack)=0.33    False
  very large 5000x5000 ran, p(crack)=0.33    False
  [PASS] T4 malformed input does not crash the app — 0 crashes of 7

Note: 'ran' on a non-egg image is not a failure of this test but IS a product gap —
the app has no reject option for images that are not candled eggs. Record it in 4.9.3
and in the limitations.
  [PASS] T4b corrupted bytes return None rather than raising — imdecode None


---
## T5–T7 · Rule base, decision logic, latency

In [6]:
# T5 rule base
rules = None
for cand in [PROC / "root_cause_rules_recovered.csv", CFG.ROOT_CAUSE_RULES_CSV]:
    if Path(cand).is_file():
        df = pd.read_csv(cand)
        rules = {(str(r["crack_pattern"]).strip(), str(r["age_class"]).strip()): r
                 for _, r in df.iterrows()}
        break
if rules:
    missing = [(p, a) for p in CFG.PATTERNS for a in CFG.AGE_ORDER if (p, a) not in rules]
    empties, bad_sum = [], []
    for k, r in rules.items():
        if not str(r.get("interpretation", "")).strip() or not str(r.get("suggested_actions", "")).strip():
            empties.append(k)
        if abs(float(r["phys_pct"]) + float(r["mech_pct"]) - 100) > 1e-6:
            bad_sum.append(k)
    record("T5a", "rule base covers all 9 pattern x age cells", not missing,
           f"{9 - len(missing)}/9")
    record("T5b", "every rule has interpretation and action text", not empties,
           f"{len(empties)} empty")
    record("T5c", "phys% + mech% = 100 in every rule", not bad_sum, f"{len(bad_sum)} violations")
else:
    record("T5a", "rule base present", False, "no rules file found")

# T6 decision logic
def classify(p4, threshold=DEFAULT_THRESHOLD, low=DEFAULT_LOW_CONF):
    pc = float(1 - p4[0]); cracked = pc >= threshold
    pat = p4[1:] / max(p4[1:].sum(), 1e-9)
    conf = float(pc * pat.max()) if cracked else float(p4[0])
    return cracked, (CFG.PATTERNS[int(pat.argmax())] if cracked else None), conf, conf < low

checks = [
    ("below threshold -> Intact", np.array([0.8, 0.1, 0.05, 0.05]), False),
    ("exactly at threshold -> cracked", np.array([0.70, 0.2, 0.05, 0.05]), True),
    ("above threshold -> cracked", np.array([0.1, 0.7, 0.1, 0.1]), True),
]
ok = True
for name, p4, expect in checks:
    cracked, pat, conf, flag = classify(p4)
    good = cracked == expect
    ok &= good
    print(f"   {name:<34} cracked={cracked} pattern={pat} conf={conf:.3f} flag={flag}  "
          f"{'ok' if good else 'MISMATCH'}")
record("T6a", "detection threshold behaves as specified", ok, f"{len(checks)} cases")
_, _, conf_lo, flag_lo = classify(np.array([0.5, 0.2, 0.15, 0.15]))
record("T6b", "uncertainty flag fires below the configured confidence",
       bool(flag_lo), f"conf {conf_lo:.3f} < {DEFAULT_LOW_CONF}")

# T7 CPU latency, end to end from a file on disk
if p is not None:
    raw = cv2.imread(str(p), cv2.IMREAD_COLOR)
    for _ in range(2):
        app_infer(preprocess_bgr(raw))
    t0 = time.time()
    N = 10
    for _ in range(N):
        app_infer(preprocess_bgr(raw))
    lat = (time.time() - t0) / N * 1000
    record("T7", "CPU latency per egg is workable on a farm laptop", bool(lat < 5000),
           f"{lat:.0f} ms end to end, {len(MODELS)}-seed ensemble")
    print(f"\n{lat:.0f} ms per egg on CPU. A 30-egg tray takes about {lat * 30 / 1000:.0f} s.")
    print("Report this, not the GPU figure, as the deployment number: the dashboard runs on CPU.")

  [PASS] T5a rule base covers all 9 pattern x age cells — 9/9
  [PASS] T5b every rule has interpretation and action text — 0 empty
  [PASS] T5c phys% + mech% = 100 in every rule — 0 violations
   below threshold -> Intact          cracked=False pattern=None conf=0.800 flag=False  ok
   exactly at threshold -> cracked    cracked=True pattern=Hairline conf=0.200 flag=True  ok
   above threshold -> cracked         cracked=True pattern=Hairline conf=0.700 flag=False  ok
  [PASS] T6a detection threshold behaves as specified — 3 cases
  [PASS] T6b uncertainty flag fires below the configured confidence — conf 0.200 < 0.6
  [PASS] T7 CPU latency per egg is workable on a farm laptop — 1285 ms end to end, 3-seed ensemble

1285 ms per egg on CPU. A 30-egg tray takes about 39 s.
Report this, not the GPU figure, as the deployment number: the dashboard runs on CPU.


---
## Summary

In [7]:
res = pd.DataFrame(RESULTS).set_index("id")
res.to_csv(TAB_DIR / "t04_functional_test_results.csv")
n_pass = int((res["result"] == "PASS").sum())
print(res.to_string())
print(f"\n{n_pass}/{len(res)} tests passed")

S = {"stage": STAGE, "generated_at": datetime.now().isoformat(timespec="seconds"),
     "n_tests": len(res), "n_passed": n_pass,
     "results": res.reset_index().to_dict("records")}
if len(pix):
    S["pixel_agreement"] = {"mean_abs_diff": round(float(pix["mean_abs_diff"].mean()), 3),
                            "min_corr": round(float(pix["corr"].min()), 5)}
if len(agree):
    S["prediction_agreement"] = {"label_agreement": round(float(agree["agree"].mean()), 4),
                                 "prob_mae": round(float(agree["prob_mae"].mean()), 5),
                                 "n": int(len(agree))}
def _safe(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    return str(o)
(OUT / "functional_test_summary.json").write_text(json.dumps(S, indent=2, default=_safe))

L = ["=" * 70, "NOTEBOOK 11 DIGEST -- FUNCTIONAL TESTING", "=" * 70,
     f"generated: {S['generated_at']}", f"{n_pass}/{len(res)} tests passed", ""]
for r in S["results"]:
    L.append(f"   [{r['result']}] {r['id']:<5s} {r['test']}")
    L.append(f"            {r['detail']}")
if "prediction_agreement" in S:
    L += ["", f"app vs notebook 07: {S['prediction_agreement']}"]
L.append("=" * 70)
digest = "\n".join(L)
(OUT / "digest.txt").write_text(digest)
print()
print(digest)
print("\nThe full table goes to an appendix. Section 5.4 needs two or three sentences only:")
print("the artefact reproduces the evaluated pipeline, and it degrades gracefully.")

                                                       test result                                          detail
id                                                                                                                
T1                   app preprocessing reproduces the cache   PASS  mean|diff| 0.68, min corr 0.9979, 0 unreadable
T2                   app reproduces notebook 07 predictions   PASS               agreement 100.0%, prob MAE 0.0034
T3a                              inference is deterministic   PASS                             max |diff| 0.00e+00
T3b         preprocessing is deterministic for a fixed seed   PASS                                  byte-identical
T4                   malformed input does not crash the app   PASS                                  0 crashes of 7
T4b         corrupted bytes return None rather than raising   PASS                                   imdecode None
T5a              rule base covers all 9 pattern x age cells   PASS              